In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# =========================================================
# NETWORK + UTILITIES
# =========================================================

def create_ring_network(n):
    adj = np.zeros((n, n))
    for i in range(n):
        adj[i][(i-1) % n] = 1
        adj[i][(i+1) % n] = 1
    return adj

def project(x):
    return np.clip(x, 0, 1)

In [ ]:
# =========================================================
# CENTRALIZED kWTA
# =========================================================

def kwta_equality(x, k, alpha=0.05, beta=0.1, steps=200):
    y = np.zeros_like(x)
    lam = 0

    y_hist, err_hist = [], []

    for _ in range(steps):
        grad = -(y - x) - lam
        y = y + alpha * grad
        lam = lam + beta * (np.sum(y) - k)

        y_hist.append(y.copy())
        err_hist.append(np.sum(y) - k)

    return np.array(y_hist), np.array(err_hist)


def kwta_inequality(x, k, alpha=0.05, beta=0.1, steps=200):
    y = np.zeros_like(x)
    lam = 0

    y_hist, err_hist = [], []

    for _ in range(steps):
        grad = -(y - x) - lam
        y = y + alpha * grad
        y = project(y)

        lam = lam + beta * (np.sum(y) - k)

        y_hist.append(y.copy())
        err_hist.append(np.sum(y) - k)

    return np.array(y_hist), np.array(err_hist)

In [ ]:
# =========================================================
# DISTRIBUTED AGENT MODEL
# =========================================================

class Agent:
    def __init__(self, idx, data, k_local, neighbors):
        self.id = idx
        self.data = data
        self.state = np.random.rand(len(data)) * 1.5

        self.k_local = k_local
        self.neighbors = neighbors

        self.lambda_val = np.random.randn() * 0.1
        self.psi_val = np.random.randn() * 0.1

    def get_neighbor_lambda(self, agents):
        return [agents[j].lambda_val for j in self.neighbors]

    def get_neighbor_psi(self, agents):
        return [agents[j].psi_val for j in self.neighbors]

    def update_equality(self, agents, lr=0.05):
        neighbor_vals = self.get_neighbor_lambda(agents)

        consensus = sum(self.lambda_val - v for v in neighbor_vals)

        grad = self.state - self.data + self.lambda_val
        self.state -= lr * grad

        self.state = project(self.state)

        error = np.sum(self.state) - self.k_local

        self.lambda_val += lr * (error - consensus)

    def update_inequality(self, agents, lr=0.05, limit=5):
        neighbor_vals = self.get_neighbor_psi(agents)

        consensus = sum(self.psi_val - v for v in neighbor_vals)

        grad = self.state - self.data + self.psi_val
        self.state -= lr * grad

        self.state = project(self.state)

        error = max(0, np.sum(self.state) - limit)

        self.psi_val += lr * (error - consensus)

In [ ]:
# =========================================================
# SIMULATION FUNCTIONS
# =========================================================

def run_distributed_equality(agents, steps=200):
    errors = []
    lambda_track = []

    for _ in range(steps):
        total_error = 0
        lambdas = []

        for agent in agents:
            agent.update_equality(agents)
            total_error += abs(np.sum(agent.state) - agent.k_local)
            lambdas.append(agent.lambda_val)

        errors.append(total_error)
        lambda_track.append(lambdas)

    return np.array(errors), np.array(lambda_track)


def run_distributed_inequality(agents, limit, steps=200):
    errors = []
    psi_track = []

    for _ in range(steps):
        total_error = 0
        psis = []

        for agent in agents:
            agent.update_inequality(agents, limit=limit)
            total_error += max(0, np.sum(agent.state) - limit)
            psis.append(agent.psi_val)

        errors.append(total_error)
        psi_track.append(psis)

    return np.array(errors), np.array(psi_track)

In [ ]:
# =========================================================
# VISUALIZATION
# =========================================================

def plot_error(err, title):
    plt.plot(err)
    plt.title(title)
    plt.xlabel("Iterations")
    plt.ylabel("Error")
    plt.grid()
    plt.show()

def plot_dual(history, title):
    for i in range(history.shape[1]):
        plt.plot(history[:, i], label=f"Agent {i}")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.show()

def plot_final(y, k, title):
    winners = np.argsort(-y)[:k]
    plt.bar(range(len(y)), y, color="gray")
    plt.bar(winners, y[winners], color="green")
    plt.title(title)
    plt.show()

In [ ]:
# =========================================================
# TEST 1: CENTRALIZED
# =========================================================

x = np.random.rand(20)
k = 5

y_eq, err_eq = kwta_equality(x, k)
y_in, err_in = kwta_inequality(x, k)

plot_error(err_eq, "Equality Constraint Error")
plot_error(err_in, "Inequality Constraint Error")

plot_final(y_eq[-1], k, "Final Equality Output")
plot_final(y_in[-1], k, "Final Inequality Output")

In [ ]:
# =========================================================
# TEST 2: DISTRIBUTED SMALL
# =========================================================

N = 6
adj = create_ring_network(N)

data = np.random.rand(20)
sizes = [4,2,3,4,5,2]
k_vals = [1,1,1,1,1,0]

agents = []
idx = 0

for i in range(N):
    local = data[idx:idx+sizes[i]]
    neighbors = [j for j in range(N) if adj[i][j] == 1]
    agents.append(Agent(i, local, k_vals[i], neighbors))
    idx += sizes[i]

errors, lambdas = run_distributed_equality(agents)

plot_error(errors, "Distributed Equality Error")
plot_dual(lambdas, "Lambda Consensus")

In [ ]:
# =========================================================
# TEST 3: DISTRIBUTED LARGE
# =========================================================

N = 10
adj = create_ring_network(N)

data = np.random.rand(1000)
sizes = [100]*N
k_vals = [1]*N

agents = []
idx = 0

for i in range(N):
    local = data[idx:idx+100]
    neighbors = [j for j in range(N) if adj[i][j] == 1]
    agents.append(Agent(i, local, k_vals[i], neighbors))
    idx += 100

errors, lambdas = run_distributed_equality(agents)

plot_error(errors, "Large Scale Error")
plot_dual(lambdas, "Large Scale Consensus")

In [ ]:
# =========================================================
# TEST 4: DISTRIBUTED INEQUALITY
# =========================================================

N = 6
adj = create_ring_network(N)

agents = [
    Agent(i, np.random.rand(3)*2, 1,
          [j for j in range(N) if adj[i][j] == 1])
    for i in range(N)
]

errors, psis = run_distributed_inequality(agents, limit=5)

plot_error(errors, "Inequality Error")
plot_dual(psis, "Psi Consensus")